In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Custom Object Detection on Neural Ocean Dataset

## Objective

The objective of this task is to train a custom YOLO model
using the Neural Ocean dataset from Roboflow.

After training, we will test the model on multiple unseen images
and evaluate its detection performance.

In [ ]:
!pip install -q ultralytics roboflow opencv-python matplotlib

In [3]:
import os
import cv2
import matplotlib.pyplot as plt

from ultralytics import YOLO
from roboflow import Roboflow
from getpass import getpass
from google.colab import files

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="w06Mz6QITM0P0KlSycgf")
project = rf.workspace("neural-ocean").project("neural_ocean")
version = project.version(3)
dataset = version.download("yolov8")
DATASET_PATH = dataset.location
                

In [9]:
print("Dataset folders:")

for folder in os.listdir(DATASET_PATH):
    print(folder)

Dataset folders:
data.yaml
README.roboflow.txt
README.dataset.txt
valid
train
test


In [10]:
data_yaml = os.path.join(DATASET_PATH, "data.yaml")

with open(data_yaml, "r") as file:
    print(file.read())

names:
- Mask
- can
- cellphone
- electronics
- gbottle
- glove
- metal
- misc
- net
- pbag
- pbottle
- plastic
- rod
- sunglasses
- tire
nc: 15
roboflow:
  license: CC BY 4.0
  project: underwater-65x5j
  url: https://universe.roboflow.com/project/underwater-65x5j/dataset/3
  version: 3
  workspace: project
test: ../test/images
train: ../train/images
val: ../valid/images



In [11]:
import yaml

with open(data_yaml, "r") as file:
    data_config = yaml.safe_load(file)

print("Number of Classes:", data_config["nc"])

print("Class Names:")

for class_id, class_name in enumerate(data_config["names"]):
    print(class_id, class_name)

Number of Classes: 15
Class Names:
0 Mask
1 can
2 cellphone
3 electronics
4 gbottle
5 glove
6 metal
7 misc
8 net
9 pbag
10 pbottle
11 plastic
12 rod
13 sunglasses
14 tire


## Dataset Description

The Neural Ocean dataset is a custom dataset
used for object detection.

It contains images and their corresponding annotations.

The dataset is downloaded in YOLO format.

The data.yaml file describes the dataset paths,
number of classes, and class names.

The training process uses the labeled images
to learn how to detect the objects.

In [13]:
model = YOLO("yolo11n.pt")

print("YOLO Nano model loaded successfully!")

YOLO Nano model loaded successfully!


In [ ]:
results = model.train(
    data=data_yaml,
    epochs=60,
    imgsz=832,
    batch=8,
     name="neural_ocean_yolo",
)

Ultralytics 8.4.218 🚀 Python-3.13.18 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/Neural_Ocean-3/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=832, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=neural_ocean_yolo-2, nbs=64, nms=None, opset=Non

## Understanding Training Parameters

### Epochs
The number of complete passes through the training dataset.

We use 60 epochs to give the model enough training time.

### Image Size
We use 832 pixels to provide more image details.

Larger image sizes may improve small object detection
but require more GPU memory.

### Optimizer
SGD is used to update the model weights during training.

### Learning Rate
The learning rate controls how much the model weights
change during each update.

### Patience
Patience controls early stopping.

If validation performance does not improve for 10 epochs,
training can stop automatically.

This helps prevent unnecessary training.

In [ ]:
results_path = "Neural_Ocean_Training/neural_ocean_yolo"

print("Training output files:")

for file in os.listdir(results_path):
    print(file)

In [ ]:
from IPython.display import Image, display

display(Image(filename=f"{results_path}/results.png"))

In [ ]:
best_model_path = f"{results_path}/weights/best.pt"

last_model_path = f"{results_path}/weights/last.pt"

best_model = YOLO(best_model_path)

print("Best model loaded successfully!")

## Understanding Model Weights

### best.pt
Contains the weights from the best validation checkpoint.

### last.pt
Contains the weights from the last training epoch.

We usually use best.pt for final prediction
because it represents the best validation performance.

Saving weights is important because we can reuse the trained model
without repeating the training process.